# Integrated CBM + GFlowNet + Rule-Regularized Neuro-Symbolic Pipeline

Notebook mỏng (launcher) chạy liên hoàn 5 giai đoạn:
1. **Stage 1b:** Huấn luyện Concept Bottleneck Model (CBM) với Concept Loss (BCE).
2. **Stage 2:** Trích xuất ma trận xác suất concept $P(c) \in [0, 1]$ từ CBM.
3. **Stage 3:** Trích xuất tập luật ngữ nghĩa bằng Random Forest ($P(\text{concept}) \to \text{Class}$).
4. **Stage 4:** GFlowNet tuyển chọn tập luật Concept tối ưu, đa dạng.
5. **Stage 5:** Huấn luyện CBM có ràng buộc luật (Rule-Regularized Fine-Tuning).

In [ ]:
# =========================================================================
# THAM SỐ THỰC NGHIỆM (Điều chỉnh tùy ý)
# =========================================================================
DATA_DIR = "/kaggle/input/vietnamese-cultural-dataset/vietnamese_cultural_dataset"
SEED = 42
BACKBONE = "mobilenetv3_small"  # mobilenetv3_small | resnet50 | efficientnet_b0 | vit_b_32
N_CONCEPTS = 30
CONCEPT_WEIGHT = 1.0
NUM_EPOCHS = 50
PATIENCE = 5
BATCH_SIZE = 64
GIT_REPO = "https://github.com/khoaddb2207532/neuro_symbolic_mlops_l2_app.git"
GIT_BRANCH = "main"

In [ ]:
# Kiểm tra dữ liệu đầu vào
from pathlib import Path
data_path = Path(DATA_DIR)
if not data_path.exists():
    # Fallback tìm kiếm trong /kaggle/input
    possible = list(Path("/kaggle/input").glob("**/*vietnamese*"))
    if possible:
        DATA_DIR = str(possible[0])
        print(f"Tự động phát hiện dữ liệu tại: {DATA_DIR}")
    else:
        print(f"CẢNH BÁO: Không tìm thấy {DATA_DIR}. Hãy kiểm tra 'Add Input' trên Kaggle.")
else:
    print(f"Dữ liệu hợp lệ tại: {DATA_DIR}")

In [ ]:
# Clone repo & setup môi trường
import subprocess
import sys
from pathlib import Path

PROJECT = Path("/kaggle/working/neuro_symbolic_mlops_l2_app")
if not PROJECT.exists():
    print("Đang clone repo...")
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO, str(PROJECT)], check=True)
print("Repo sẵn sàng tại:", PROJECT)

In [ ]:
# Cài đặt thư viện phụ thuộc
%cd /kaggle/working/neuro_symbolic_mlops_l2_app
!pip install -q torchgfn tensordict dvclive dvc openpyxl

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# Cập nhật file params.yaml từ tham số notebook
import yaml

with open("params.yaml", "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)

params["data_dir"] = DATA_DIR
params["seed"] = SEED
params["num_epochs"] = NUM_EPOCHS
params["patience"] = PATIENCE
params["batch_size"] = BATCH_SIZE
params["baseline_comparison"]["selected_architecture"] = BACKBONE

# Thiết lập CBM làm nguồn đặc trưng và mục tiêu phạt luật
params.setdefault("cbm", {})
params["cbm"]["n_concepts"] = N_CONCEPTS
params["cbm"]["concept_weight"] = CONCEPT_WEIGHT
params.setdefault("features", {})
params["features"]["source"] = "cbm"
params.setdefault("rule_penalty", {})
params["rule_penalty"]["target_model"] = "cbm"

with open("params.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(params, f, sort_keys=False, allow_unicode=True)

print("Cấu hình params.yaml đã sẵn sàng cho CBM + GFlowNet!")

In [ ]:
# Chạy toàn bộ pipeline tích hợp 5 Stage
!python -m pipelines.run_cbm_neuro_symbolic_pipeline --config params.yaml

In [ ]:
# Hiển thị Top luật Concept được GFlowNet tuyển chọn (Người đọc hiểu 100%)
import pandas as pd
from pathlib import Path

rules_excel = Path("outputs/04_filtered_rules/selected_rules.xlsx")
if rules_excel.exists():
    df = pd.read_excel(rules_excel)
    print(f"\n=== TỔNG CỘNG CÓ {len(df)} LUẬT CONCEPT ĐƯỢC GFLOWNET CHỌN ===")
    display(df.head(15))
else:
    print("Chưa tìm thấy selected_rules.xlsx")

In [ ]:
# Đóng gói kết quả đầu ra thành file zip để tải về
!zip -r -q /kaggle/working/cbm_neuro_symbolic_outputs.zip outputs/
print("\nĐÃ NÉN TOÀN BỘ KẾT QUẢ TẠI: /kaggle/working/cbm_neuro_symbolic_outputs.zip")
!ls -lh /kaggle/working/cbm_neuro_symbolic_outputs.zip